# Phase 2: Predictive Modeling (Frequency & Severity)

In this notebook, we build the core actuarial models. We will predict:
1. **Claim Frequency** (`ClaimNb` offset by `Exposure`) using Poisson Deviances.
2. **Claim Severity** (`AvgSeverity` for policies with claims) using Gamma Deviances.

We benchmark simple Generalized Linear Models (GLMs) against advanced LightGBM gradient boosting models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
import lightgbm as lgb
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance, mean_absolute_error
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

## 1. Claim Frequency Modeling
We start by predicting the number of claims per year.

In [ ]:
# Load Data
df = pd.read_csv('../data/processed/features.csv')
features = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Density", "Region"]
X = df[features]
y = df["ClaimNb"]
w = df["Exposure"]

# Train/Test Split
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=0.2, random_state=42)

# Preprocessor
categorical_cols = ["Area", "VehBrand", "VehGas", "Region"]
numeric_cols = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_cols)
    ]
)

### Baseline: Scikit-learn Poisson Regressor

In [ ]:
# Train baseline
pipeline_base = Pipeline([
    ("preprocessor", preprocessor),
    ("model", PoissonRegressor(alpha=1e-4, max_iter=300))
])
pipeline_base.fit(X_train, y_train, model__sample_weight=w_train)

# Evaluate
preds_base = pipeline_base.predict(X_test)
preds_base = pd.Series(preds_base).clip(lower=1e-6)
dev_base = mean_poisson_deviance(y_test, preds_base, sample_weight=w_test)
print(f"Baseline Poisson Deviance: {dev_base:.4f}")

### Advanced: LightGBM Poisson Regressor

In [ ]:
# Train LightGBM
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

lgb_model = lgb.LGBMRegressor(
    objective="poisson", n_estimators=100, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1
)
lgb_model.fit(X_train_prep, y_train, sample_weight=w_train)

preds_lgb = lgb_model.predict(X_test_prep)
preds_lgb = pd.Series(preds_lgb).clip(lower=1e-6)
dev_lgb = mean_poisson_deviance(y_test, preds_lgb, sample_weight=w_test)
print(f"LightGBM Poisson Deviance: {dev_lgb:.4f}")
print(f"Improvement: {((dev_base - dev_lgb) / dev_base) * 100:.2f}%")

**Insight**: LightGBM captures non-linear relationships and interactions better than the baseline GLM, resulting in a lower (better) Poisson Deviance.

## 2. Claim Severity Modeling
Next, we model the average severity of claims. We only look at policies with at least 1 claim.

In [ ]:
# Filter claims > 0
df_claims = df[(df["ClaimNb"] > 0) & (df["AvgSeverity"] > 0)]
X_sev = df_claims[features]
y_sev = df_claims["AvgSeverity"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sev, y_sev, test_size=0.2, random_state=42)

# Preprocessor (without dropping first for tree models)
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(sparse_output=False), categorical_cols)
    ]
)
X_train_s_prep = preprocessor_tree.fit_transform(X_train_s)
X_test_s_prep = preprocessor_tree.transform(X_test_s)

### Advanced: LightGBM Gamma Regressor

In [ ]:
lgb_sev = lgb.LGBMRegressor(
    objective="gamma", n_estimators=100, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1
)
lgb_sev.fit(X_train_s_prep, y_train_s)

preds_sev_lgb = lgb_sev.predict(X_test_s_prep)
preds_sev_lgb = np.clip(preds_sev_lgb, a_min=1e-3, a_max=None)

gamma_dev_lgb = mean_gamma_deviance(y_test_s, preds_sev_lgb)
mae_log_lgb = mean_absolute_error(np.log1p(y_test_s), np.log1p(preds_sev_lgb))

print(f"LightGBM Gamma Deviance: {gamma_dev_lgb:.4f}")
print(f"LightGBM Log-MAE: {mae_log_lgb:.4f}")

**Conclusion**: We have successfully modeled both the frequency and severity of claims. The LightGBM model proves to be a strong candidate for modern actuarial pricing algorithms due to its ability to capture complex feature interactions automatically.